Only optmize and no z ordering on same nyc taxi data

In [0]:
%fs
ls /databricks-datasets/nyctaxi/tables/nyctaxi_yellow/

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/_delta_log/,_delta_log/,0,1788165691045
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,374549044,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet,189069652,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet,373889711,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet,376396848,1605327443000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,364876497,1605327444000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet,363016752,1605327465000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet,203737885,1605327471000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,part-00001-d61d36d1-b864-4dc2-b80b-92e7d1515232-c001.snappy.parquet,365327633,1605327484000
dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,part-00002-0e3744e2-5c15-4c60-baee-bd45cf02a686-c000.snappy.parquet,355258373,1605327487000


In [0]:
%sql
describe formatted delta.`dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow`

col_name,data_type,comment
vendor_id,string,null
pickup_datetime,timestamp,null
dropoff_datetime,timestamp,null
passenger_count,int,null
trip_distance,double,null
pickup_longitude,double,null
pickup_latitude,double,null
rate_code_id,int,null
store_and_fwd_flag,string,null
dropoff_longitude,double,null


In [0]:
%sql
drop table nyc_taxi;

In [0]:
%sql
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1'

Load the table with 5 parq files from above but split it to 200 files
Took only 10 files, are the whole table is huge. 

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:5]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

In [0]:
%sql
select count (*) from nyc_taxi

count(*)
45142478


This query has to open about 113 files out of 200 to get the results. Pruning is very less

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(*)
158


Trip distance field is not stored in ordered manner , which makes the query in efficient. Do optimize on it, with no z ordering

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,169.8,part-00130-8851df80-94ba-4cb2-9104-c281c815e388.c000.zstd.parquet
0.0,45.0,part-00028-88e82d79-7c8a-4478-87a1-64020538ffaa.c000.zstd.parquet
0.0,177.4,part-00184-43d98d45-50f0-4f61-aed2-842df72a2e6f.c000.zstd.parquet
0.0,180.7,part-00058-d11c2e62-8b0f-4b75-a971-f804a4531436.c000.zstd.parquet
0.0,49.2,part-00135-29951076-32d1-4480-89a4-bd8d80a7c962.c000.zstd.parquet
0.0,601.5,part-00049-5fda9bb8-780e-49c4-bda3-17d1ff11fd34.c000.zstd.parquet
0.0,1000000.0,part-00162-7a3d5544-6d20-4e93-a052-858095dabfd0.c000.zstd.parquet
0.0,88.0,part-00010-b05db485-7d52-4a81-84a6-960406eeeeed.c000.zstd.parquet
0.0,177.7,part-00061-a15c4f7e-25aa-4daa-8000-82a98276a9fa.c000.zstd.parquet
0.0,46.9,part-00006-54cc0d43-d07f-465b-ab6d-7e36ed9209e2.c000.zstd.parquet


OPTIMIZE WITH NO ZORDER

In [0]:
%sql
optimize nyc_taxi 

path,metrics
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1,"List(5, 200, List(197187245, 368646752, 3.341866424E8, 5, 1670933212), List(5994912, 6082174, 6026038.81, 200, 1205207762), 0, null, null, 0, 1, 200, 0, true, 0, 0, 1788187742526, 1788187757042, 24, 5, null, List(0, 0), null, 18, 18, 25355, 0, null, null, 0)"


Optimize stats;
- numFilesAdded: 5
- numFilesRemoved: 200

Re run the query

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(*)
158


After Optmize alone
- number of files read	5
 

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,8000004.9,part-00001-d8d9fa59-a8e1-48c0-8725-f57801c9ef96.c000.snappy.parquet
0.0,6746333.1,part-00002-7b911aa8-2023-4b25-999d-a2c345c1fba6.c000.snappy.parquet
0.0,1.18000006E7,part-00003-444bca88-fb26-4fd1-a52e-acdcf5470d18.c000.snappy.parquet
0.0,1.18000003E7,part-00000-47648a13-5664-4132-b3d6-144657db9b78.c000.snappy.parquet
0.0,9025000.0,part-00004-ca1f493f-c1d6-4561-aacc-5bd3a6de0f00.c000.snappy.parquet


With out z order, though number of files reduces to 5 , they still have overlapping ranges of trip_distance